# Assignment 02 — Application 1: Diabetes Prediction

**Course:** Intelligent System Development  
**Assignment:** From Data Representation to a Deployable Intelligent System

### Objective
Build a multiclass diabetes prediction system following:

**Raw Data → Understand → Clean → Represent → Learn → Evaluate → Persist**

The assignment requires the data representation to be explicitly explained before model training. The selected dataset contains 1,000 patient records and 14 variables. The `CLASS` target has three classes: `N`, `P`, and `Y`.

> `N` = Non-Diabetic, `P` = Predict-Diabetic, `Y` = Diabetic, according to descriptions of this dataset family. The exact label meaning should be stated cautiously in the final report because the supplied CSV itself only contains the labels.


## 1. Dataset and Problem Definition

**X = patient features**  
**y = diabetes class**

This is a **multiclass classification** problem.

We remove `ID` and `No_Pation` from the model features because they are identifiers rather than clinical measurements. `Gender` is categorical; the remaining selected variables are numerical.

The assignment requires reporting the original dataframe shape, feature-matrix shape, data types, encoding/scaling, and final model-input shape.


In [ ]:
# 2. Import libraries

import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

import joblib

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("Libraries loaded.")


In [ ]:
# 3. Load dataset

# Put the CSV in the same folder as this notebook, or change DATA_PATH.
DATA_PATH = "Dataset of Diabetes .csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
# 4. First inspection

display(df.head())

print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nInfo:")
df.info()


In [ ]:
# 5. Descriptive statistics

display(df.describe(include="all").T)


In [ ]:
# 6. Missing values and duplicates

print("Missing values:")
display(df.isna().sum())

print("\nTotal missing values:", df.isna().sum().sum())

print("\nExact duplicated rows:", df.duplicated().sum())


In [ ]:
# 7. Inspect categorical values and target distribution

print("Gender values:")
display(df["Gender"].value_counts(dropna=False))

print("\nCLASS values:")
display(df["CLASS"].value_counts(dropna=False))


### Data-quality observations

The supplied dataset contains small inconsistencies in categorical text, such as lowercase `f` in `Gender` and whitespace variants in `CLASS`. These should be normalized before modeling.

The `CLASS` distribution is strongly imbalanced, so accuracy alone is not enough. Macro F1, recall, precision and the confusion matrix are important for evaluating the minority classes.


In [ ]:
# 8. Data cleaning

clean_df = df.copy()

# Normalize string columns
clean_df["Gender"] = clean_df["Gender"].astype("string").str.strip().str.upper()
clean_df["CLASS"] = clean_df["CLASS"].astype("string").str.strip().str.upper()

# Convert numeric columns explicitly
numeric_candidates = [
    "ID", "No_Pation", "AGE", "Urea", "Cr", "HbA1c",
    "Chol", "TG", "HDL", "LDL", "VLDL", "BMI"
]

for col in numeric_candidates:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# Remove exact duplicates after normalization
before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
removed_duplicates = before - len(clean_df)

print("Rows before cleaning:", before)
print("Exact duplicates removed:", removed_duplicates)
print("Rows after cleaning:", len(clean_df))

print("\nGender after normalization:")
display(clean_df["Gender"].value_counts(dropna=False))

print("\nCLASS after normalization:")
display(clean_df["CLASS"].value_counts(dropna=False))

print("\nMissing values after cleaning:")
display(clean_df.isna().sum())


In [ ]:
# 9. Invalid-value checks

# Basic plausibility checks for the available numerical variables.
invalid_checks = {
    "AGE <= 0": (clean_df["AGE"] <= 0).sum(),
    "BMI <= 0": (clean_df["BMI"] <= 0).sum(),
    "Urea < 0": (clean_df["Urea"] < 0).sum(),
    "Cr < 0": (clean_df["Cr"] < 0).sum(),
    "HbA1c < 0": (clean_df["HbA1c"] < 0).sum(),
    "Chol < 0": (clean_df["Chol"] < 0).sum(),
    "TG < 0": (clean_df["TG"] < 0).sum(),
    "HDL < 0": (clean_df["HDL"] < 0).sum(),
    "LDL < 0": (clean_df["LDL"] < 0).sum(),
    "VLDL < 0": (clean_df["VLDL"] < 0).sum(),
}

display(pd.Series(invalid_checks, name="count"))


In [ ]:
# 10. Outlier analysis using the IQR rule

numeric_features = [
    "AGE", "Urea", "Cr", "HbA1c", "Chol",
    "TG", "HDL", "LDL", "VLDL", "BMI"
]

outlier_report = []

for col in numeric_features:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((clean_df[col] < lower) | (clean_df[col] > upper)).sum()
    outlier_report.append([col, q1, q3, lower, upper, count])

outlier_report = pd.DataFrame(
    outlier_report,
    columns=["Feature", "Q1", "Q3", "Lower bound", "Upper bound", "Outlier count"]
)

display(outlier_report)

print(
    "\nOutliers are reported rather than automatically deleted. "
    "Medical measurements can contain legitimate extreme values, so "
    "removing them without domain evidence may discard useful information."
)


## 11. Exploratory Data Analysis

The assignment asks for meaningful visualizations including target distribution, important feature distributions, relationships with the target, and correlation analysis.

For each important figure, the report should contain:
- **Observation:** what the figure shows;
- **Interpretation:** what it means;
- **ML implication:** why it matters.


In [ ]:
# 11.1 Target distribution

plt.figure(figsize=(7, 4))
sns.countplot(data=clean_df, x="CLASS", order=clean_df["CLASS"].value_counts().index)
plt.title("Diabetes Class Distribution")
plt.xlabel("Diabetes Class")
plt.ylabel("Number of Patients")
plt.show()


In [ ]:
# 11.2 Important numerical feature distributions

features_to_plot = ["AGE", "HbA1c", "BMI", "Chol"]

for feature in features_to_plot:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=clean_df, x=feature, hue="CLASS", kde=True, element="step")
    plt.title(f"{feature} Distribution by Diabetes Class")
    plt.show()


In [ ]:
# 11.3 Feature vs target relationships

for feature in ["AGE", "HbA1c", "BMI", "Urea", "Cr"]:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=clean_df, x="CLASS", y=feature)
    plt.title(f"{feature} by Diabetes Class")
    plt.show()


In [ ]:
# 11.4 Correlation analysis

corr = clean_df[numeric_features + ["ID", "No_Pation"]].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.show()


## 12. Data Representation

The raw CSV is first represented as a Pandas DataFrame.

For modeling:

**CSV → DataFrame → cleaned table → feature matrix → encoded/scaled matrix → model input**

Let:

- `N` = number of cleaned observations;
- `d` = number of final encoded/scaled features.

Before one-hot encoding, the model uses 11 biological/demographic features:

`Gender, AGE, Urea, Cr, HbA1c, Chol, TG, HDL, LDL, VLDL, BMI`

`Gender` is categorical and is converted using one-hot encoding. Numerical features are standardized using `StandardScaler`.

The preprocessing is placed inside a scikit-learn `Pipeline` so that the same transformations are learned from the training data and reused during validation, testing and deployment. This avoids preprocessing leakage.


In [ ]:
# 13. Define X and y

# Remove identifier columns from the ML representation.
feature_columns = [
    "Gender", "AGE", "Urea", "Cr", "HbA1c",
    "Chol", "TG", "HDL", "LDL", "VLDL", "BMI"
]

target_column = "CLASS"

X = clean_df[feature_columns].copy()
y = clean_df[target_column].copy()

print("X shape before preprocessing:", X.shape)
print("y shape:", y.shape)

print("\nExample raw feature vector:")
display(X.head(1))


In [ ]:
# 14. Train / validation / test split: 70% / 15% / 15%

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nClass distribution:")
print("Train:")
display(y_train.value_counts(normalize=True).sort_index())
print("Validation:")
display(y_val.value_counts(normalize=True).sort_index())
print("Test:")
display(y_test.value_counts(normalize=True).sort_index())


In [ ]:
# 15. Preprocessing pipeline

numeric_features = [
    "AGE", "Urea", "Cr", "HbA1c", "Chol",
    "TG", "HDL", "LDL", "VLDL", "BMI"
]

categorical_features = ["Gender"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Show the final numerical representation.
X_train_transformed = preprocessor.fit_transform(X_train)
print("Final training matrix shape:", X_train_transformed.shape)
print("Final data type:", X_train_transformed.dtype)


In [ ]:
# 16. Example of final numerical representation

feature_names = preprocessor.get_feature_names_out()

representation_example = pd.DataFrame(
    X_train_transformed[:5],
    columns=feature_names
)

display(representation_example)

print("Final feature dimension d =", X_train_transformed.shape[1])


## 17. Model Development

Five models are compared:

1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Tree
4. Random Forest
5. Support Vector Machine (SVM)

The comparison uses the validation set. The test set is kept untouched until the final selected model is evaluated.


In [ ]:
# 17. Define models

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),
    "SVM": SVC(
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_SEED
    )
}

pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

print("Models prepared:", list(pipelines.keys()))


In [ ]:
# 18. Train and evaluate on validation set

def evaluate_classifier(model, X_data, y_data):
    pred = model.predict(X_data)

    result = {
        "Accuracy": accuracy_score(y_data, pred),
        "Precision_macro": precision_score(y_data, pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_data, pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_data, pred, average="macro", zero_division=0),
        "Precision_weighted": precision_score(y_data, pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_data, pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_data, pred, average="weighted", zero_division=0)
    }

    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_data)
            result["ROC_AUC_ovr_macro"] = roc_auc_score(
                y_data, proba, multi_class="ovr", average="macro"
            )
        except Exception:
            result["ROC_AUC_ovr_macro"] = np.nan
    else:
        result["ROC_AUC_ovr_macro"] = np.nan

    return result

validation_results = []
fitted_models = {}

for name, pipeline in pipelines.items():
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    metrics = evaluate_classifier(pipeline, X_val, y_val)
    metrics["Model"] = name
    metrics["Training Time (s)"] = elapsed

    validation_results.append(metrics)
    fitted_models[name] = pipeline

validation_df = pd.DataFrame(validation_results).set_index("Model")
validation_df = validation_df.sort_values("F1_macro", ascending=False)

display(validation_df.round(4))


### Model selection rule

Because the dataset is imbalanced and contains a minority `P` class, **Macro F1** is used as the primary validation criterion. Macro F1 gives each class equal importance instead of allowing the majority class to dominate the score.

The final selection should also consider interpretability, robustness, training cost and deployment simplicity, as required by the assignment.


In [ ]:
# 19. Select the best model using validation Macro F1

best_model_name = validation_df.index[0]
best_validation_f1 = validation_df.loc[best_model_name, "F1_macro"]

print("Selected model:", best_model_name)
print("Validation Macro F1:", round(best_validation_f1, 4))


In [ ]:
# 20. Refit selected pipeline on Train + Validation, then evaluate once on Test

X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", models[best_model_name])
])

final_pipeline.fit(X_train_final, y_train_final)

test_metrics = evaluate_classifier(final_pipeline, X_test, y_test)

print("Final model:", best_model_name)
display(pd.DataFrame([test_metrics]).round(4))


In [ ]:
# 21. Classification report and confusion matrix

y_pred = final_pipeline.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.show()


## 22. Error Analysis

The confusion matrix should be interpreted rather than only displayed.

Pay particular attention to:
- false negatives for the diabetic class `Y`;
- confusion between `P` and `Y`;
- whether the minority `P` class is being ignored;
- whether high accuracy is caused mainly by the large `Y` class.

The final report should discuss these observations using the actual confusion matrix produced above.


In [ ]:
# 23. Model persistence

MODEL_DIR = "diabetes_model"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "diabetes_pipeline.joblib")

# Save the complete preprocessing + model pipeline.
joblib.dump(final_pipeline, MODEL_PATH)

print("Saved:", MODEL_PATH)
print("File exists:", os.path.exists(MODEL_PATH))


In [ ]:
# 24. Reload saved model and test inference

loaded_pipeline = joblib.load(MODEL_PATH)

sample = X_test.iloc[[0]]
prediction = loaded_pipeline.predict(sample)[0]

print("Input sample:")
display(sample)

print("Predicted class:", prediction)

if hasattr(loaded_pipeline, "predict_proba"):
    probabilities = loaded_pipeline.predict_proba(sample)[0]
    classes = loaded_pipeline.classes_

    probability_table = pd.DataFrame({
        "Class": classes,
        "Probability": probabilities
    }).sort_values("Probability", ascending=False)

    display(probability_table)


In [ ]:
# 25. Save a compact model-comparison table

comparison_path = os.path.join(MODEL_DIR, "validation_model_comparison.csv")
validation_df.to_csv(comparison_path)

print("Saved comparison table:", comparison_path)


## 26. Report-ready Summary

Fill the following values from the outputs above when writing the report:

- Original dataset shape: **1,000 × 14**
- Cleaned dataset shape: obtained from `clean_df.shape`
- Input features before encoding: **11**
- Numerical features: **10**
- Categorical features: **1 (`Gender`)**
- Target: **`CLASS`**
- Classes: **N, P, Y**
- Train/validation/test split: **70% / 15% / 15%**
- Final representation: **encoded + standardized feature matrix**
- Final feature dimension: obtained from `X_train_transformed.shape[1]`
- Models compared: **5**
- Primary selection metric: **Macro F1**
- Final model: obtained from `best_model_name`
- Final test metrics: obtained from `test_metrics`
- Persisted artifact: **`diabetes_model/diabetes_pipeline.joblib`**

### Representation statement

A single patient is represented initially as one CSV/DataFrame row. After removing identifiers, cleaning categorical values, one-hot encoding `Gender`, imputing missing values if necessary, and standardizing numerical variables, the row becomes a numerical feature vector. A batch of patients forms a feature matrix:

**X ∈ R^(N × d)**

where `N` is the number of patients and `d` is the number of final numerical features supplied to the classifier.


## 27. Reproducibility

- Python version: check with `platform.python_version()`
- Random seed: `42`
- Dataset: supplied diabetes CSV
- Split: stratified 70/15/15
- Preprocessing: median imputation + StandardScaler + OneHotEncoder
- Models: Logistic Regression, KNN, Decision Tree, Random Forest, SVM
- Saved pipeline: `diabetes_model/diabetes_pipeline.joblib`

The same saved pipeline must be used by the future Web API so that deployment performs exactly the same preprocessing as training.


In [ ]:
# 28. Environment information

import platform
import sklearn

print("Python:", platform.python_version())
print("OS:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
